<div align="center">

 #
###

---

**AI Hachathon**
 Track RAG — Group 11 - K
---

### Team Members           

| Name                   
|:---|                 
| Mostafa Sayed            
| Mariam Magdy             
| Rawan Ahmed              
| Nour  Yasser              

---

---

###  supervisor

| Name|  
|:---|   
|ENG: Rawan Mahmuod   |
|ENG: Moaz Gehad      |
|ENG: Mohamed Walid   |
          

---

</div>


## 1. Install dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')
!pip install -q pypdf sentence-transformers chromadb rank-bm25 openai gradio numpy tqdm

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Configuration

In [ ]:
import os
from google.colab import userdata

# ---------- Paths ----------
DATASET_PATH = "/content/drive/MyDrive/MEd"
PDF_FILES = ["ehae178.pdf"]

# ---------- ChromaDB ----------
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "med_pdfs"

# ---------- Chunking ----------
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300

# ---------- Retrieval ----------
TOP_K_DENSE = 15
TOP_K_SPARSE = 15
CANDIDATE_K = 30
TOP_K_FINAL = 4
USE_RERANKER = True

# ---------- Embedding model (multilingual, handles English + Arabic) ----------
EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"

# ---------- Reranker (cross-encoder) ----------
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ---------- Context window for generation ----------
MAX_CONTEXT_CHARS = 12000

# ---------- OpenRouter ----------
# Falls back to an env var if you're not running in Colab / no userdata secret set.
try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = None
OPENROUTER_API_KEY = OPENROUTER_API_KEY or os.environ.get("OPENROUTER_API_KEY")

OPENROUTER_MODEL = "meta-llama/llama-3.3-70b-instruct"

if not OPENROUTER_API_KEY:
    print("⚠️  No OPENROUTER_API_KEY found. Set it via Colab secrets (userdata) "
          "or the OPENROUTER_API_KEY environment variable before generating answers.")


## 4. Extract text from PDFs

Extracts text page-by-page so every chunk keeps a reference to its source file and page number.

In [ ]:
from pathlib import Path
from pypdf import PdfReader
import hashlib
import re

PDF_DIR = Path(DATASET_PATH)

# extracted segment (and later every chunk) can carry a "section" label like
# "Introduction", "2.3 Diagnosis", "Recommendations", etc.
SECTION_KEYWORDS = {
    "abstract", "introduction", "background", "methods", "materials and methods",
    "results", "discussion", "conclusion", "conclusions", "recommendations",
    "references", "acknowledgements", "acknowledgments", "limitations", "summary",
    "objectives", "diagnosis", "treatment", "management", "epidemiology",
    "definitions", "clinical presentation", "risk factors", "screening",
    "follow-up", "evidence", "appendix", "key points", "gaps in evidence",
}
_HEADING_NUMBERED_RE = re.compile(r"^\d+(\.\d+)*\.?\s+[A-Z][A-Za-z0-9\s\-/,&()]{2,80}$")
_HEADING_ALLCAPS_RE = re.compile(r"^[A-Z][A-Z0-9\s\-/,&()]{2,60}$")

# Class of Recommendation (I, IIa, IIb, III) / Level of Evidence (A, B, C, D)
# table labels -- e.g. "I A", "IIa B", "III C" -- show up as short, all-caps
# lines in ESC recommendation boxes and were being misread as section
# headings. They must be excluded explicitly since they otherwise satisfy
# the all-caps heading pattern below.
_COR_TOKENS = {"I", "II", "III", "IIA", "IIB"}
_LOE_TOKENS = {"A", "B", "C", "D"}


def is_cor_loe_label(line: str) -> bool:
    """True for stray Class-of-Recommendation / Level-of-Evidence labels
    like "I A" or "IIa B", which are table cells, not headings."""
    tokens = line.strip().split()
    if not tokens or len(tokens) > 2:
        return False
    return all(t.upper() in _COR_TOKENS or t.upper() in _LOE_TOKENS for t in tokens)


def looks_like_heading(line: str) -> bool:
    """Cheap heuristic: does this line look like a section heading?"""
    line = line.strip()
    if not line or len(line) > 90:
        return False
    if is_cor_loe_label(line):
        return False
    if _HEADING_NUMBERED_RE.match(line):
        return True
    words = line.split()
    # Require >=2 words for the all-caps case, so lone acronyms like "CVD"
    # or "ABPM" that wrap onto their own line aren't mistaken for headings.
    if _HEADING_ALLCAPS_RE.match(line) and 2 <= len(words) <= 8:
        return True
    if line.lower().strip(" .:") in SECTION_KEYWORDS:
        return True
    return False


def make_doc_id(filename: str) -> str:
    """Short, stable id for a source file (same file -> same id every run)."""
    return hashlib.md5(filename.encode("utf-8")).hexdigest()[:8]


def get_title(reader: PdfReader, filename: str) -> str:
    """Prefer the PDF's own metadata title; fall back to a cleaned filename."""
    try:
        meta_title = (reader.metadata.title or "").strip() if reader.metadata else ""
    except Exception:
        meta_title = ""
    if meta_title:
        return meta_title
    stem = Path(filename).stem
    stem = re.sub(r"[_\-]+", " ", stem).strip()
    return stem.title() if stem else filename


def extract_section_segments(pdf_path: Path):
    """
    Walk the PDF page by page and line by line, splitting the text every time
    a new section heading is detected. Returns a list of dicts:
    {source, doc_id, title, page, section, text}

    Each dict is the text found under one heading *on one page*. If a section
    continues onto the next page, that page produces another segment with the
    same "section" label — these get stitched back together in
    `merge_into_sections` below, which is what lets chunking respect section
    boundaries instead of page boundaries.
    """
    reader = PdfReader(str(pdf_path))
    doc_id = make_doc_id(pdf_path.name)
    title = get_title(reader, pdf_path.name)
    segments = []
    current_section = "Unknown section"

    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ""
        buf = []
        for line in raw.split("\n"):
            if looks_like_heading(line):
                if buf:
                    text = "\n".join(buf).strip()
                    if text:
                        segments.append({
                            "source": pdf_path.name,
                            "doc_id": doc_id,
                            "title": title,
                            "page": i,
                            "section": current_section,
                            "text": text,
                        })
                    buf = []
                current_section = line.strip()
                continue
            buf.append(line)

        if buf:
            text = "\n".join(buf).strip()
            if text:
                segments.append({
                    "source": pdf_path.name,
                    "doc_id": doc_id,
                    "title": title,
                    "page": i,
                    "section": current_section,
                    "text": text,
                })
    return segments


def merge_into_sections(segments):
    """
    Stitch consecutive segments that share the same (source, section) into a
    single section block, tracking the first/last page it appears on. This
    turns a per-page list of segments into a per-section list, so chunking
    (next cell) can operate on whole sections instead of raw pages.
    """
    blocks = []
    for seg in segments:
        if (
            blocks
            and blocks[-1]["source"] == seg["source"]
            and blocks[-1]["section"] == seg["section"]
        ):
            blocks[-1]["text"] += " " + seg["text"]
            blocks[-1]["page_end"] = seg["page"]
        else:
            blocks.append({
                "source": seg["source"],
                "doc_id": seg["doc_id"],
                "title": seg["title"],
                "section": seg["section"],
                "page_start": seg["page"],
                "page_end": seg["page"],
                "text": seg["text"],
            })
    return blocks


all_segments = []
for fname in PDF_FILES:
    fpath = PDF_DIR / fname
    if not fpath.exists():
        print(f"⚠️  Missing file: {fpath}")
        continue
    file_segments = extract_section_segments(fpath)
    all_segments.extend(file_segments)
    print(f"{fname}: extracted {len(file_segments)} raw page/section segments")

print(f"\nTotal raw segments extracted: {len(all_segments)}")
if not all_segments:
    raise SystemExit("No text extracted. Check DATASET_PATH and PDF_FILES above, "
                     "and that the PDFs are in that Drive folder.")

all_sections = merge_into_sections(all_segments)
print(f"Total section blocks (after stitching across pages): {len(all_sections)}")


ehae178.pdf: extracted 366 raw page/section segments

Total raw segments extracted: 366
Total section blocks (after stitching across pages): 284


In [ ]:
# Inspect the first extracted section block across ALL files.
if all_sections:
    print("--- First extracted section block ---")
    s = all_sections[0]
    print("Source:", s["source"])
    print("Section:", s["section"])
    print("Pages:", f'{s["page_start"]}-{s["page_end"]}')
    print("Text preview:", s["text"][:400])
else:
    print("No sections extracted — check DATASET_PATH and PDF_FILES above.")


--- First extracted section block ---
Source: ehae178.pdf
Section: 2024 ESC Guidelines for the management of
Pages: 1-1
Text preview: elevated blood pressure and hypertension 
Developed by the task force on the management of elevated blood pressure and 
hypertension of the European Society of Cardiology (ESC) and endorsed by the 
European Society of Endocrinology (ESE) and the European Stroke Organisation (ESO) 
Authors/Task Force Members: John William McEvoy  
 *
†
, (Chairperson) (Ireland), 
Cian P. McCarthy  
‡
, (Task Force 


## 5. Clean and chunk text

In [ ]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

documents = []
doc_id_counter = 0
for section_index, sec in enumerate(all_sections):
    cleaned = clean_text(sec["text"])
    section_chunks = chunk_text(cleaned)
    for chunk_index, c in enumerate(section_chunks):
        documents.append({
            "id": str(doc_id_counter),
            "source": sec["source"],
            "doc_id": sec.get("doc_id", ""),
            "title": sec.get("title", sec["source"]),
            "page": sec["page_start"],          # kept for backward-compat
            "page_start": sec["page_start"],
            "page_end": sec["page_end"],
            "section": sec.get("section", "Unknown section"),
            "section_index": section_index,
            "chunk_index": chunk_index,
            "chunks_in_section": len(section_chunks),
            "text": c,
        })
        doc_id_counter += 1

print(f"Total sections processed: {len(all_sections)}")
print(f"Total chunks created: {len(documents)}")
if documents:
    print("\nExample chunk:")
    print(documents[0])


Total sections processed: 284
Total chunks created: 1071

Example chunk:
{'id': '0', 'source': 'ehae178.pdf', 'doc_id': '28e14505', 'title': 'Ehae178', 'page': 1, 'page_start': 1, 'page_end': 1, 'section': '2024 ESC Guidelines for the management of', 'section_index': 0, 'chunk_index': 0, 'chunks_in_section': 9, 'text': 'elevated blood pressure and hypertension Developed by the task force on the management of elevated blood pressure and hypertension of the European Society of Cardiology (ESC) and endorsed by the European Society of Endocrinology (ESE) and the European Stroke Organisation (ESO) Authors/Task Force Members: John William McEvoy * † , (Chairperson) (Ireland), Cian P. McCarthy ‡ , (Task Force Co-ordinator) (United States of America), Rosa Maria Bruno ‡ , (Task Force Co-ordinator) (France), Sofie Brouwers (Belgium), Michelle D. Canavan (Ireland), Claudio Ceconi (Italy), Ruxandra Maria Christodorescu (Romania), Stella S. Daskalopoulou (Canada), Charles J. Ferro 1 (United Kingdo

In [ ]:
import re
_CITATION_PAREN_RE = re.compile(r"\s*\([^()]*\.pdf[^()]*\)", re.IGNORECASE)
_CITATION_BRACKET_RE = re.compile(r"\s*\[[^\[\]]*\.pdf[^\[\]]*\]", re.IGNORECASE)
# Bare inline mentions with no brackets at all, e.g. "ehae178.pdf, page 12"
_CITATION_BARE_RE = re.compile(
    r"\s*[\w\-]+\.pdf(?:\s*,\s*(?:p\.?|page|pages|section)[^.,;\n]*)*",
    re.IGNORECASE,
)


def clean_generated_answer(text: str) -> str:
    """
    Strip inline source citations (filename / page / section mentions) from
    the LLM's generated answer text before it's shown in the chat.

    The Evidence Sources panel on the right already carries this
    information with full metadata (source, page, section, score), so
    repeating it inline in the prose is redundant and clutters the answer.
    This is a safety net -- SYSTEM_PROMPT already instructs the model not
    to add these, but models don't always follow instructions perfectly.
    """
    if not text:
        return text

    cleaned = _CITATION_PAREN_RE.sub("", text)
    cleaned = _CITATION_BRACKET_RE.sub("", cleaned)
    cleaned = _CITATION_BARE_RE.sub("", cleaned)

    # Tidy up whitespace/punctuation left behind by the removal, e.g.
    # "target range .pdf) ." -> "target range."
    cleaned = re.sub(r"[ \t]+([.,;:])", r"\1", cleaned)   # "word ." -> "word."
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)            # collapse double spaces
    cleaned = re.sub(r"\n[ \t]+", "\n", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    return cleaned.strip()


## 6. Build embeddings + ChromaDB collection

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL_NAME)

def embed_texts(texts, is_query=False):
    prefixed = [("query: " if is_query else "passage: ") + t for t in texts]
    embs = embedder.encode(prefixed, normalize_embeddings=True, show_progress_bar=True)
    return embs.tolist()

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Fresh collection each run (drop if it already exists, so re-runs don't duplicate)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if documents:
    corpus_texts = [d["text"] for d in documents]
    corpus_embeddings = embed_texts(corpus_texts, is_query=False)

    collection.add(
        ids=[d["id"] for d in documents],
        embeddings=corpus_embeddings,
        documents=corpus_texts,
        metadatas=[{
            "source": d["source"],
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "section": d["section"],
        } for d in documents],
    )
    print(f"ChromaDB collection built: {collection.count()} vectors")
else:
    print("No documents to index — check DATASET_PATH and PDF_FILES, then re-run from Section 4.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

ChromaDB collection built: 1071 vectors


## 7. Build BM25 sparse index (for hybrid retrieval)

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    """Tokenize text for BM25 retrieval."""
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(doc["text"]) for doc in documents]

if tokenized_corpus:
    bm25 = BM25Okapi(tokenized_corpus)
    # Map document ID -> position in `documents`
    id_to_idx = {doc["id"]: idx for idx, doc in enumerate(documents)}
    print("BM25 index built.")
    print(f"Documents indexed: {len(tokenized_corpus)}")
else:
    bm25 = None
    id_to_idx = {}
    print("No documents available for BM25.")


BM25 index built.
Documents indexed: 1071


In [ ]:
print(documents[0])

{'id': '0', 'source': 'ehae178.pdf', 'doc_id': '28e14505', 'title': 'Ehae178', 'page': 1, 'page_start': 1, 'page_end': 1, 'section': '2024 ESC Guidelines for the management of', 'section_index': 0, 'chunk_index': 0, 'chunks_in_section': 9, 'text': 'elevated blood pressure and hypertension Developed by the task force on the management of elevated blood pressure and hypertension of the European Society of Cardiology (ESC) and endorsed by the European Society of Endocrinology (ESE) and the European Stroke Organisation (ESO) Authors/Task Force Members: John William McEvoy * † , (Chairperson) (Ireland), Cian P. McCarthy ‡ , (Task Force Co-ordinator) (United States of America), Rosa Maria Bruno ‡ , (Task Force Co-ordinator) (France), Sofie Brouwers (Belgium), Michelle D. Canavan (Ireland), Claudio Ceconi (Italy), Ruxandra Maria Christodorescu (Romania), Stella S. Daskalopoulou (Canada), Charles J. Ferro 1 (United Kingdom), Eva Gerdts (Norway), Henner Hanssen (Switzerland), Julie Harris (Unit

## 8. Hybrid retrieval (dense + sparse, fused with RRF)

Reciprocal Rank Fusion combines both rankings without needing to normalize scores
across different scales.

In [ ]:
#semantic search
def dense_search(query: str, top_k: int = TOP_K_DENSE) -> list[int]:
    """Retrieve documents using dense vector similarity. Returns indices into `documents`."""
    query_embedding = embed_texts([query], is_query=True)
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    retrieved_ids = results["ids"][0]
    return [id_to_idx[doc_id] for doc_id in retrieved_ids if doc_id in id_to_idx]

#Exact Search
def sparse_search(query: str, top_k: int = TOP_K_SPARSE) -> list[int]:
    """Retrieve documents using BM25. Returns indices into `documents`."""
    if bm25 is None:
        return []
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    return [int(idx) for idx in top_indices]


def reciprocal_rank_fusion(rank_lists: list[list[int]], k: int = 60) -> list[int]:
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.
        score(d) = sum(1 / (k + rank))
    Returns documents ordered by fused score (best first).
    """
    scores: dict[int, float] = {}
    for ranked_list in rank_lists:
        for rank, doc_idx in enumerate(ranked_list, start=1):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank)

    ranked_documents = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [doc_idx for doc_idx, _score in ranked_documents]


def hybrid_search(
    query: str,
    top_k_dense: int = TOP_K_DENSE,
    top_k_sparse: int = TOP_K_SPARSE,
    candidate_k: int = CANDIDATE_K,
) -> list[int]:
    """
    Hybrid retrieval:  Dense + BM25  -->  RRF  -->  candidate documents.
    """
    dense_results = dense_search(query, top_k=top_k_dense)
    sparse_results = sparse_search(query, top_k=top_k_sparse)

    fused = reciprocal_rank_fusion([dense_results, sparse_results])

    return fused[:candidate_k]


## 9. Cross-encoder reranking (optional but recommended)

In [ ]:
from sentence_transformers import CrossEncoder

if USE_RERANKER:
    reranker = CrossEncoder(RERANKER_MODEL_NAME)
    print(f"Reranker loaded: {RERANKER_MODEL_NAME}")
else:
    reranker = None
    print("Reranker disabled.")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
def rerank(query: str, candidate_indices: list[int], top_k: int = TOP_K_FINAL) -> list[dict]:
    """
    Rerank candidate documents using a cross-encoder.
    The CrossEncoder returns raw relevance scores; we also expose a 0-1 display
    score (raw / 10, clamped) alongside the raw score.
    """
    if not candidate_indices:
        return []

    if not USE_RERANKER or reranker is None:
        return [documents[idx].copy() for idx in candidate_indices[:top_k]]

    pairs = [(query, documents[idx]["text"]) for idx in candidate_indices]
    raw_scores = reranker.predict(pairs)

    ranked = sorted(zip(candidate_indices, raw_scores), key=lambda x: x[1], reverse=True)

    results = []
    for doc_idx, raw_score in ranked[:top_k]:
        normalized_score = max(0.0, min(1.0, float(raw_score) / 10.0))
        result = documents[doc_idx].copy()
        result["reranker_raw_score"] = float(raw_score)
        result["reranker_score"] = normalized_score
        results.append(result)

    return results


## 10. Complete retrieval pipeline

In [ ]:
def retrieve(
    query: str,
    top_k_dense: int = TOP_K_DENSE,
    top_k_sparse: int = TOP_K_SPARSE,
    candidate_k: int = CANDIDATE_K,
    final_k: int = TOP_K_FINAL,
) -> list[dict]:
    """
    Query -> Dense + BM25 -> RRF -> candidate set -> cross-encoder rerank -> final results.
    """
    candidates = hybrid_search(
        query=query,
        top_k_dense=top_k_dense,
        top_k_sparse=top_k_sparse,
        candidate_k=candidate_k,
    )
    return rerank(query=query, candidate_indices=candidates, top_k=final_k)


## 11. Sanity-check retrieval (optional)

"source": d["source"],
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "section": d["section"],

In [ ]:
query = "What are the recommended blood pressure treatment targets for adults receiving antihypertensive therapy according to the 2024 ESC Guidelines?"

results = retrieve(query)
print(f"Query: {query}")
print(f"Retrieved {len(results)} documents\n")

for rank, result in enumerate(results, start=1):
    print("=" * 100)
    print(f"RESULT #{rank}")
    print("=" * 100)
    print("Source:", result["source"])
    print("Document ID:", result["id"])
    print("Title:", result["title"])
    page_start = result.get("page_start", result.get("page"))
    page_end = result.get("page_end", result.get("page"))
    page_display = str(page_start) if page_start == page_end else f"{page_start}-{page_end}"
    print("Page:", page_display)
    print("Section:", result["section"])
    print("Chunk:", f'{result.get("chunk_index", 0) + 1}/{result.get("chunks_in_section", 1)}')
    if "reranker_score" in result:
        print("Reranker score:", result["reranker_score"])
    print("\nText:")
    print(result["text"][:1500])
    print()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: What are the recommended blood pressure treatment targets for adults receiving antihypertensive therapy according to the 2024 ESC Guidelines?
Retrieved 4 documents

RESULT #1
Source: ehae178.pdf
Document ID: 411
Title: Ehae178
Page: 53-54
Section: treatment
Chunk: 2/11
Reranker score: 0.6428735256195068

Text:
Supplementary data online). Persons with elevated BP who receive treatment are also recommended to achieve a target of 120–129/70–79 mmHg. Therefore, the treatment target in the 2024 Guidelines is always 120–129/70–79 mmHg (but only if treatment is tolerated and with certain exceptions where more lenient targets are advised). In contrast, the treatment threshold may differ based on CVD risk, specifically in the elevated BP category. For example, in addition to hypertensive adults with BP ≥140/90 mmHg, there are individuals with an office systolic BP of 130–139 mmHg and/or diastolic BP of 80–89 mmHg who have sufficiently high CVD risk to recommend BP-lowering drug treatment

## 12. System prompt

In [ ]:
SYSTEM_PROMPT = """
You are a precise medical research assistant. Answer ONLY from the
provided context excerpts from the source PDFs.

RULES:
- If the user's question asks for the name(s) of a medication/drug/treatment
  (e.g. "what medicine", "which drug", "name of treatment", "write name of
  medicant") for any condition, DO NOT answer it — even if the context
  contains drug names. Respond only with:
  "I can't provide medication names. Please consult a licensed healthcare
  professional or refer to the source document directly."
- Do not list, enumerate, or partially reveal drug/medication names in any
  form (brand name, generic name, or drug class) under any framing of the
  question, including indirect ones like "what is used to treat X" or
  "what options exist for X".
- Never use outside knowledge, assumptions, or memory.
- Never invent diagnoses, treatments, doses, contraindications.
- If the context is insufficient, say so clearly.
- If sources conflict, mention the conflict.
- Do not infer COR or LOE unless explicitly stated.
- Preserve all conditions and qualifiers such as "if tolerated",
  "should be considered", and "may be considered".
- Distinguish diagnosis/evaluation, treatment, target, COR, and LOE.

CITATIONS:
- Every claim you make MUST be grounded in the provided context — never
  answer from outside knowledge.
- Do NOT write inline citations in your answer: no filenames, no "(page
  X)", no "(Section Y)", no ".pdf" mentions, no source names anywhere in
  the prose. The exact source, page, and section for every retrieved
  passage is already shown to the user separately in an Evidence Sources
  panel, so repeating it inline is redundant.
- Just answer the clinical question in plain prose/bullets, grounded in
  the context, without naming or pointing to sources yourself.

STYLE:
- Answer directly and concisely.
- Use short headings and bullets for complex answers.
- Avoid unnecessary repetition.

FINAL CHECK:
Before answering, first check whether the question is asking for a
medication/drug name. If yes, use the refusal response above and stop.
Otherwise, verify that every clinical claim is supported by the provided
context. Do not add any inline citation, filename, page, or section
reference to the answer text — the retrieved context is the only source
of truth, and its provenance is shown separately, not in your prose.
"""


## 13. Context formatting

In [ ]:
def format_context(results: list[dict], max_chars: int = MAX_CONTEXT_CHARS) -> str:
    """
    Format retrieved documents into structured context for the LLM, stopping
    before the total exceeds `max_chars` so the prompt stays within budget.
    """
    if not results:
        return "NO RELEVANT CONTEXT WAS RETRIEVED."

    context_parts = []
    total_chars = 0

    for i, result in enumerate(results, start=1):
        source = result.get("source", "source")
        doc_id = result.get("doc_id", "id")
        title = result.get("title", "title")
        page_start = result.get("page_start", result.get("page", "page"))
        page_end = result.get("page_end", page_start)
        page_display = str(page_start) if page_start == page_end else f"{page_start}-{page_end}"
        section = result.get("section", "section")
        text = result.get("text", "").strip()
        reranker_score = result.get("reranker_score")

        block = f"--- CONTEXT {i} ---\n\nTITLE: {title}\nSOURCE: {source}\nPAGE: {page_display}\nSECTION: {section}\n"
        if reranker_score is not None:
            block += f"RETRIEVAL SCORE: {reranker_score:.4f}\n"
        block += f"\nTEXT:\n{text}"
        block = block.strip()

        if total_chars + len(block) > max_chars:
            break

        context_parts.append(block)
        total_chars += len(block)

    return "\n\n".join(context_parts)


## 14. Generation via OpenRouter

In [ ]:
from openai import OpenAI

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)


def generate_answer(
    query: str,
    context: str,
    chat_history: list[dict] | None = None,
    model: str = OPENROUTER_MODEL,
    temperature: float = 0.3,
) -> str:
    """
    Calls the LLM with the system prompt + retrieved context + question.
    `chat_history` is an optional list of {"role": "user"/"assistant", "content": str}
    from earlier turns, so follow-up questions keep conversational context
    (retrieval itself still runs fresh on the latest query).
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if chat_history:
        messages.extend(chat_history)

    user_turn = f"CONTEXT:\n{context}\n\nQUESTION:\n{query}"
    messages.append({"role": "user", "content": user_turn})

    response = openrouter_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


## 15. End-to-end RAG query

"source": d["source"],
            "doc_id": d["doc_id"],
            "title": d["title"],
            "page": d["page"],
            "section": d["section"],

In [ ]:
def rag_query(
    query: str,
    chat_history: list[dict] | None = None,
    verbose: bool = True,
) -> dict:
    """
    Full pipeline:
    retrieve -> format context -> generate -> clean answer.

    Source metadata is preserved separately for the Evidence Sources UI.
    """

    results = retrieve(query)

    # Context used by the LLM
    context = format_context(results)

    # Generate answer
    raw_answer = generate_answer(query, context)

    # Remove citations ONLY from generated answer
    answer = clean_generated_answer(raw_answer)

    # Keep all evidence metadata for the UI
    sources = [
        {
            "source": r.get("source"),
            "doc_id": r.get("id"),
            "title": r.get("title"),
            "page_start": r.get("page_start", r.get("page")),
            "page_end": r.get("page_end", r.get("page")),
            "section": r.get("section"),
            "chunk_index": r.get("chunk_index"),
            "chunks_in_section": r.get("chunks_in_section"),
            "reranker_score": r.get("reranker_score"),
        }
        for r in results
    ]

    if verbose:
        print("=" * 100)
        print("QUESTION:", query)
        print("=" * 100)
        print(answer)

        print("\nSOURCES USED:")
        for s in sources:
            page_display = (
                str(s["page_start"])
                if s["page_start"] == s["page_end"]
                else f'{s["page_start"]}-{s["page_end"]}'
            )

            score = (
                f"{s['reranker_score']:.4f}"
                if s["reranker_score"] is not None
                else "N/A"
            )

            print(
                f"  Document ID: {s['doc_id']}  "
                f"Source: {s['source']}  "
                f"Page: {page_display}  "
                f"Section: {s['section']}  "
                f"Title: {s['title']}  "
                f"Reranker score: {score}"
            )

    return {
        "answer": answer,
        "sources": sources
    }

## 16. Evaluation Metrics

In [ ]:
import numpy as np

# Define example evaluation queries and their expected relevant sections/documents.
# In a real scenario, this would be a carefully curated dataset.
# For simplicity, we'll map to document sources and pages.

eval_queries = [
    {
        "query": "What are the recommended blood pressure treatment targets for adults receiving antihypertensive therapy?",
        "relevant_sources": [
            {"source": "ehae178.pdf", "pages": [53, 54]} # Updated page numbers based on manual rag_query output
        ]
    },
    {
        "query": "What is the definition of hypertension and how is it classified?",
        "relevant_sources": [
            {"source": "ehae178.pdf", "pages": [6, 7]} # Keep as-is for now, might need further adjustment
        ]
    }
]

print("Defined example evaluation queries.")

Defined example evaluation queries.


In [ ]:

_ = rag_query("What are the recommended blood pressure treatment targets for adults receiving antihypertensive therapy according to the 2024 ESC Guidelines?")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION: What are the recommended blood pressure treatment targets for adults receiving antihypertensive therapy according to the 2024 ESC Guidelines?
The recommended blood pressure treatment target for adults receiving antihypertensive therapy is 120–129/70–79 mmHg, but only if treatment is tolerated and with certain exceptions where more lenient targets are advised. 

In persons with diabetes who are receiving blood pressure-lowering drugs, it is recommended to target systolic BP to 120–129 mmHg, if tolerated.

SOURCES USED:
  Document ID: 411  Source: ehae178.pdf  Page: 53-54  Section: treatment  Title: Ehae178  Reranker score: 0.6429
  Document ID: 216  Source: ehae178.pdf  Page: 25  Section: 6.1. Definition and classification of  Title: Ehae178  Reranker score: 0.4951
  Document ID: 525  Source: ehae178.pdf  Page: 64-65  Section: 9.7.1. Relationship between hypertension and chronic  Title: Ehae178  Reranker score: 0.3962
  Document ID: 526  Source: ehae178.pdf  Page: 64-65  Secti

### Retrieval Evaluation

In [ ]:
def evaluate_retrieval(query_data: list, k: int = TOP_K_FINAL) -> dict:
    """Evaluates retrieval performance using Hit Rate@K."""
    hits = 0
    total_queries = len(query_data)

    print(f"\nEvaluating retrieval for {total_queries} queries (Hit Rate@{k})...")

    for i, item in enumerate(query_data):
        query = item["query"]
        expected_relevant = item["relevant_sources"]

        # Retrieve documents using the RAG pipeline's retrieval part
        retrieved_docs = retrieve(query, final_k=k)

        # Check if any of the retrieved documents are among the expected relevant ones
        is_hit = False
        for retrieved_doc in retrieved_docs:
            retrieved_source = retrieved_doc["source"]
            retrieved_page_start = retrieved_doc["page_start"]
            retrieved_page_end = retrieved_doc["page_end"]

            for relevant_item in expected_relevant:
                if retrieved_source == relevant_item["source"]:
                    # A hit occurs if any of the expected relevant pages are within the retrieved document's page range
                    for expected_page in relevant_item["pages"]:
                        if retrieved_page_start <= expected_page <= retrieved_page_end:
                            is_hit = True
                            break
                if is_hit:
                    break
            if is_hit:
                break

        if is_hit:
            hits += 1

        print(f"Query {i+1}/{total_queries}: {'HIT' if is_hit else 'MISS'}")

    hit_rate = hits / total_queries
    return {"hit_rate": hit_rate}

# Run the evaluation
eval_results = evaluate_retrieval(eval_queries, k=TOP_K_FINAL)
print(f"\nRetrieval Hit Rate@{TOP_K_FINAL}: {eval_results['hit_rate']:.4f}")


Evaluating retrieval for 2 queries (Hit Rate@4)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 1/2: HIT


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 2/2: MISS

Retrieval Hit Rate@4: 0.5000


In [ ]:
print("\n--- Retrieval Hit Rate for various K values ---")
for k_val in range(1, TOP_K_FINAL + 1):
    eval_results_k = evaluate_retrieval(eval_queries, k=k_val)
    print(f"Retrieval Hit Rate@{k_val}: {eval_results_k['hit_rate']:.4f}")


--- Retrieval Hit Rate for various K values ---

Evaluating retrieval for 2 queries (Hit Rate@1)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 1/2: HIT


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 2/2: MISS
Retrieval Hit Rate@1: 0.5000

Evaluating retrieval for 2 queries (Hit Rate@2)...


...

Query 2/2: MISS
Retrieval Hit Rate@3: 0.5000

Evaluating retrieval for 2 queries (Hit Rate@4)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 1/2: HIT


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query 2/2: MISS
Retrieval Hit Rate@4: 0.5000


---

# Website

Everything above builds the pipeline. Everything below serves it as a website.
Run all cells top to bottom; the last one prints your link.

### Install the server

In [ ]:
!pip install -q fastapi uvicorn nest_asyncio

### The page

The whole interface — HTML, CSS, JavaScript. Edit here, re-run this cell, refresh
the browser. Colours are the `:root` block near the top.

In [ ]:
import base64, os
HTML_PATH = "index.html"
html_b64 = "..."
with open(HTML_PATH, "wb") as f:
    f.write(base64.b64decode(html_b64))
print(f"{HTML_PATH} written ({os.path.getsize(HTML_PATH):,} bytes)")

index.html written (43,144 bytes)


### The server

In [ ]:
  import os, re, sys, time, socket, threading, subprocess

PORT = 8000

# ---- check the pipeline
_need = ["retrieve", "format_context", "generate_answer", "documents", "PDF_FILES"]
_gone = [n for n in _need if n not in globals()]
if _gone:
    raise SystemExit(
        "Missing from the notebook: " + ", ".join(_gone) +
        "\nRun the notebook top to bottom first, then run this cell."
    )
if not documents:
    raise SystemExit(
        "`documents` is empty — the PDFs were never indexed.\n"
        "Re-run the extraction and ChromaDB cells first."
    )
print(f"Pipeline: {len(documents):,} chunks from {len(PDF_FILES)} documents")

# ---------------------------------------------------------------- deps
try:
    import fastapi, uvicorn, nest_asyncio  # noqa
except ImportError:
    print("Installing fastapi, uvicorn, nest_asyncio...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastapi", "uvicorn", "nest_asyncio"], check=True)

from fastapi import FastAPI
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn, nest_asyncio

nest_asyncio.apply()

# ---------------------------------------------------------------- api
app = FastAPI(title="ESC Evidence")
# Allows index.html to be opened from your desktop and still reach this server.
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])


class Ask(BaseModel):
    question: str
    history: list = []


@app.get("/", response_class=HTMLResponse)
def _page():
    # Re-read every request, so editing index.html + refreshing is enough.
    return open(HTML_PATH).read()


@app.get("/api/status")
def _status():
    return {"documents": len(PDF_FILES), "chunks": len(documents)}


def _source_payload(r, citation_id):
    return {
        "citation_id": citation_id,
        "source": r.get("source"),
        "doc_id": r.get("id"),
        "title": r.get("title"),
        "page": r.get("page", r.get("page_start")),
        "page_start": r.get("page_start", r.get("page")),
        "page_end": r.get("page_end", r.get("page")),
        "section": r.get("section"),
        "section_index": r.get("section_index"),
        "chunk_index": r.get("chunk_index"),
        "chunks_in_section": r.get("chunks_in_section"),
        "reranker_score": (
            round(float(r.get("reranker_score")), 4)
            if r.get("reranker_score") is not None else None
        ),
        "text": r.get("text", ""),
    }


@app.post("/api/chat")
def _chat(a: Ask):
    try:
        t0 = time.time()
        results = retrieve(a.question)

        # Generate only from retrieved context. Keep the source provenance
        # separate so the patient UI can display exact citations/chunks.
        raw_answer = generate_answer(
            a.question,
            format_context(results),
            chat_history=a.history or None,
        )
        answer = clean_generated_answer(raw_answer) if "clean_generated_answer" in globals() else raw_answer

        sources = [_source_payload(r, i + 1) for i, r in enumerate(results)]

        return {
            "answer": answer,
            "sources": sources,
            "elapsed": round(time.time() - t0, 1),
            "source_count": len(sources),
        }
    except Exception as e:
        import traceback; traceback.print_exc()
        return JSONResponse(
            status_code=500,
            content={
                "answer": f"Server error: {e}",
                "sources": [],
            },
        )


# ---------------------------------------------------------------- serve
def _up(p):
    with socket.socket() as s:
        s.settimeout(0.4)
        return s.connect_ex(("127.0.0.1", p)) == 0


if _up(PORT):
    print(f"Server:   already running on :{PORT}, reusing it")
else:
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="error"),
        daemon=True,
    ).start()
    for _ in range(30):
        if _up(PORT):
            break
        time.sleep(0.4)
    else:
        raise SystemExit("Server failed to start. Restart the runtime, run all, retry.")
    print(f"Server:   up on :{PORT}")

Pipeline: 1,071 chunks from 1 documents
Server:   already running on :8000, reusing it


### Get the link

Prints a public URL. Open it — that's the website. Keep this Colab tab open.

In [ ]:
# ---- public link
if not os.path.exists("cloudflared"):
    print("Tunnel:   downloading...")
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared",
        shell=True, check=True,
    )

_p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

URL, _t0 = None, time.time()
for _l in _p.stdout:
    _m = re.search(r"https://[-\w]+\.trycloudflare\.com", _l)
    if _m:
        URL = _m.group(0)
        break
    if time.time() - _t0 > 60:
        break

print()
if URL:
    print("=" * 64)
    print("  OPEN THIS IN YOUR BROWSER")
    print(f"  {URL}")
    print("=" * 64)
    print("  Keep the Colab tab open — closing it kills the link.")
    print("  Edited index.html? Just refresh the browser, no re-run needed.")
else:
    print("  Tunnel didn't start. Run this cell again.")


  OPEN THIS IN YOUR BROWSER
  https://justin-dam-occurred-article.trycloudflare.com
  Keep the Colab tab open — closing it kills the link.
  Edited index.html? Just refresh the browser, no re-run needed.
